# RUL Prediction — Phase 1, Notebook 3
## C-MAPSS FD001: Baseline Models & Evaluation Metrics

**Dissertation:** Deep Learning-Based Remaining Useful Life Prediction for Industrial Rotating Equipment
**Author:** K Vatsav Sai Ram · BITS ID 2024TM05021

---

### Why baselines first?
A deep-learning model is only worth its complexity if it **beats simpler models**. So before the LSTM,
we establish the "numbers to beat" with two classical regressors — **Random Forest** and **XGBoost**.

This notebook also introduces the **two evaluation metrics** we use for the rest of the project:
1. **RMSE** — standard regression error (symmetric).
2. **PHM Scoring Function** — the C-MAPSS competition metric, which *penalises late predictions more
   than early ones* (predicting more life than is left is dangerous). This directly encodes the
   "false negatives cost more" design decision from the viva.

By the end we have baseline RMSE and PHM scores that Notebook 4's LSTM must improve on.


## 1. Setup & load the preprocessed arrays

We load the `.npz` produced by Notebook 2 — no reprocessing needed.

In [ ]:
import numpy as np, os, json
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
plt.rcParams["figure.figsize"] = (10, 4)

# --- project paths: Colab+Drive if available, else a local folder ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/RUL_Dissertation"
except ImportError:
    PROJECT_DIR = os.path.abspath(".")

DATA_DIR     = os.path.join(PROJECT_DIR, "data", "cmapss")
ARTIFACT_DIR = os.path.join(PROJECT_DIR, "artifacts")
PROC_DIR     = os.path.join(PROJECT_DIR, "processed")
MODEL_DIR    = os.path.join(PROJECT_DIR, "models")
for _d in (ARTIFACT_DIR, PROC_DIR, MODEL_DIR):
    os.makedirs(_d, exist_ok=True)
assert os.path.isdir(DATA_DIR), f"Place the C-MAPSS .txt files in {DATA_DIR}"
print("DATA_DIR:", DATA_DIR)

data = np.load(os.path.join(PROC_DIR, "fd001_windows.npz"))
X_train, y_train = data["X_train"], data["y_train"]
X_test,  y_test  = data["X_test"],  data["y_test"]
WINDOW   = int(data["window"]); N_FEAT = int(data["n_features"])
print("Loaded:")
print("  X_train", X_train.shape, "y_train", y_train.shape)
print("  X_test ", X_test.shape,  "y_test ", y_test.shape)
print(f"  window={WINDOW}, features={N_FEAT}")

## 2. The evaluation metrics

### 2.1 RMSE
Root Mean Squared Error — average magnitude of prediction error, in cycles. Lower is better.

### 2.2 PHM Scoring Function
The official C-MAPSS metric. For each engine let \(d = \hat{\text{RUL}} - \text{RUL}\) (predicted − true):

$$s = \sum_i \begin{cases} e^{-d_i/13} - 1 & d_i < 0 \;(\text{early prediction})\\ e^{\;d_i/10} - 1 & d_i \ge 0 \;(\text{late prediction}) \end{cases}$$

The asymmetry matters: a **late** prediction (\(d>0\), model thinks more life remains than actually does)
grows with base \(e^{1/10}\), while an **early** prediction shrinks with \(e^{-1/13}\). So late predictions
are punished harder — exactly the maintenance logic that missing a failure is costlier than a false alarm.
Lower score is better; unlike RMSE it is **unbounded** and dominated by the worst late predictions.

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

def phm_score(y_true, y_pred):
    d = y_pred - y_true                       # positive = LATE (bad), negative = EARLY
    s = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return float(np.sum(s))

# quick illustration of the asymmetry
for d in [-10, 10]:
    kind = "early" if d < 0 else "late"
    pen = (np.exp(-d/13)-1) if d < 0 else (np.exp(d/10)-1)
    print(f"error of {d:+d} cycles ({kind:5s}) -> penalty {pen:5.2f}")
print("Same magnitude, but the LATE error is penalised more heavily.")

## 3. Prepare inputs for classical models

Random Forest and XGBoost can't read a 3-D `(samples, window, features)` tensor — they need a flat
2-D table. The simplest, standard approach is to **flatten** each window into one long feature vector
of length `window × features` (30 × 14 = 420 features).

*(This deliberately throws away the explicit time-ordering — which is precisely the weakness the LSTM
will exploit to beat these baselines.)*

In [ ]:
X_train_flat = X_train.reshape(X_train.shape[0], -1)   # (17731, 420)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)        # (100, 420)
print("Flattened:", X_train_flat.shape, "|", X_test_flat.shape)

## 4. Baseline A — Random Forest

A robust, low-tuning ensemble of decision trees. Good first baseline.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, max_depth=None, n_jobs=-1, random_state=SEED)
rf.fit(X_train_flat, y_train)

rf_pred = rf.predict(X_test_flat)
rf_pred = np.clip(rf_pred, 0, None)     # RUL can't be negative
print(f"Random Forest  ->  RMSE = {rmse(y_test, rf_pred):.2f}   PHM score = {phm_score(y_test, rf_pred):.1f}")

## 5. Baseline B — XGBoost

Gradient-boosted trees — usually stronger than RF on tabular data. If `xgboost` isn't present in the
runtime we install it.

In [ ]:
try:
    from xgboost import XGBRegressor
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"])
    from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1,
)
xgb.fit(X_train_flat, y_train)

xgb_pred = np.clip(xgb.predict(X_test_flat), 0, None)
print(f"XGBoost        ->  RMSE = {rmse(y_test, xgb_pred):.2f}   PHM score = {phm_score(y_test, xgb_pred):.1f}")

## 6. Results summary & diagnostic plots

We collect the baseline numbers into a table (this goes straight into your mid-sem report), and plot
predicted-vs-true RUL to see *where* the models struggle — typically at high RUL (healthy engines),
which the cap already helps with.

In [ ]:
import pandas as pd
results = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost"],
    "RMSE": [rmse(y_test, rf_pred), rmse(y_test, xgb_pred)],
    "PHM_Score": [phm_score(y_test, rf_pred), phm_score(y_test, xgb_pred)],
}).round(2)
print(results.to_string(index=False))

# persist for later comparison against the LSTM
results.to_csv(os.path.join(ARTIFACT_DIR, "baseline_results_FD001.csv"), index=False)
print("\nSaved baseline_results_FD001.csv")

In [ ]:
# Predicted vs true RUL (sorted by true RUL for readability)
order = np.argsort(y_test)
fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for a, (name, pred) in zip(ax, [("Random Forest", rf_pred), ("XGBoost", xgb_pred)]):
    a.plot(y_test[order], "o-", color="#10243B", ms=3, label="true RUL")
    a.plot(pred[order], "o", color="#B5651D", ms=3, alpha=0.7, label="predicted")
    a.set_title(f"{name}: predicted vs true (test engines)")
    a.set_xlabel("test engine (sorted by true RUL)"); a.legend()
ax[0].set_ylabel("RUL (cycles)")
plt.tight_layout(); plt.show()

## 7. Summary & next step

**Baselines established (FD001):** Random Forest and XGBoost RMSE + PHM scores are saved. These are the
numbers the LSTM must beat to justify the deep-learning approach.

For reference, with this windowed + capped setup the tree baselines on FD001 typically land around
**RMSE ≈ 15–18**, and a good LSTM/CNN reaches **RMSE ≈ 12–15**. The margin looks small in RMSE but the
LSTM's real advantage shows more clearly in the **PHM score** (fewer dangerous late predictions), since
it exploits the time-ordering the flattened baselines discard. Exact numbers vary slightly with settings.

**Next — Notebook 4 (LSTM):** build the recurrent model that uses the *time-ordering* the baselines
ignored, train it on GPU, and compare RMSE + PHM against these baselines. That comparison is the core
result of your mid-semester report.
